# TP 1

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np

In [ ]:
food_parquet = '../../data/food.parquet'
parquet_file = pq.ParquetFile(food_parquet)

In [ ]:
parquet_columns = ["code", "brands", "product_name", "nutriments", "categories_tags", "nutriscore_score"]
first_batch = next(parquet_file.iter_batches(5_000_000, columns=parquet_columns))
df = first_batch.to_pandas()
df.to_parquet("../../data/food_column_filtered.parquet") 
# 2:45 sur MacOS (16Go)

In [ ]:
# df = pd.read_parquet("../../data/food.parquet")

df_france = df[df['countries_tags'].astype(str).str.lower().str.contains('france', na=False)]

df_france

In [ ]:
# Combien de produits sont vendus en France ?
print(f"Nombre de produits vendus en Monde : {len(df)}")
print(f"Nombre de produits vendus en France : {len(df_france)}")
set_nutriscore = []
for nutriscore in df_france['nutriscore_grade'] :
    set_nutriscore.append(nutriscore)
print(set(set_nutriscore))

In [ ]:
# Quelle part du catalogue possède un Nutri-Score renseigné ?
nutriscore_grade = ['a', 'b', 'c', 'd', 'e']
count = 0
for nutriscore in df_france['nutriscore_grade']:
    if nutriscore in nutriscore_grade :
        count += 1
part_nutriscore = round((100 * count)/len(df_france), 2)
print(f"Part des Nutriscore renseigné : {part_nutriscore}%")

In [ ]:
# Quelles sont les dix marques les plus présentes ?
brands = {}
for brand in df_france['brands'] :
    brand_str = str(brand)
    if brand_str.lower() not in brands:
        brands[brand_str.lower()] = 1
    else:
        brands[brand_str.lower()] += 1 

brands_sorted = dict(sorted(brands.items(), key=lambda item: item[1], reverse=True))
top_10_brands = dict(list(brands_sorted.items())[:10])
print(top_10_brands)

In [ ]:
# Quel est le taux de valeurs manquantes sur les nutriments clés (energy_100g, sugars_100g, salt_100g) ?

nutriments = round(df_france['nutriments'].isna().mean()*100, 2)
print(f"Part des nutriments non remplis : {nutriments}%")

def is_missing(list_nutriments, name_nutriment):
    # Si la case entière est vide (NaN, None, float au lieu d'une liste...)
    if not isinstance(list_nutriments, (list, np.ndarray)):
        return True
        
    # On parcourt chaque dictionnaire de la liste
    for nutriment in list_nutriments:
        # Si on trouve le bon nutriment (ex: 'energy')
        if isinstance(nutriment, dict) and nutriment.get('name') == name_nutriment:
            valeur = nutriment.get('100g')
            # On vérifie si sa valeur est vide (None, NaN, ou chaîne vide)
            if valeur is None or pd.isna(valeur) or valeur == '':
                return True
            else:
                return False # On a trouvé une valeur valide !
                
    # Si on a fini la liste et qu'on n'a pas trouvé le nutriment, il est manquant
    return True

total_lines = len(df_france)

# On applique la fonction pour chaque nutriment ciblé
# (le .sum() va additionner tous les True, donc compter les manquants)
energy_missing = df_france['nutriments'].apply(lambda x: is_missing(x, 'energy')).sum()
sugars_missing = df_france['nutriments'].apply(lambda x: is_missing(x, 'sugars')).sum()
salt_missing = df_france['nutriments'].apply(lambda x: is_missing(x, 'salt')).sum()

# On calcule le taux (en pourcentage)
taux_energy = (energy_missing / total_lines) * 100
taux_sugars = (sugars_missing / total_lines) * 100
taux_salt = (salt_missing / total_lines) * 100

# Affichage propre des résultats
print("Taux de valeurs manquantes :")
print(f"- Énergie : {taux_energy:.2f} % ({energy_missing} manquants sur {total_lines})")
print(f"- Sucres  : {taux_sugars:.2f} % ({sugars_missing} manquants sur {total_lines})")
print(f"- Sel     : {taux_salt:.2f} % ({salt_missing} manquants sur {total_lines})")

In [ ]:
# Selon toi, qu'est-ce qui semble le plus « sale » ou atypique dans ces données ?
# La colonne 'brands', car ses données sont soit NaN, soit vide, soit se répète, un léger changement d'orthographe qui par conséquent se considère comme unique. 
print(df.columns)

# TP 2

In [ ]:
# Vérifier rapidement si la colonne de codes-barres contient des doublons 
print(df['code'].duplicated().value_counts()) #Nous sommes toujours sur le _light.parquet

# Filtrer DataFrame pour isoler les lignes où les valeurs de sucres sont physiquement impossibles

def verif_value_100g(list_nutriments):
    if not isinstance(list_nutriments, (list, np.ndarray)):
        return False
        
    for nutriment in list_nutriments:
        if isinstance(nutriment, dict) and nutriment.get('name') in ['sugars','energy']:
            valeur = nutriment.get('100g')
            if valeur is not None and not pd.isna(valeur) and valeur != '':
                try : 
                    valeur_num = float(valeur)
                    if nutriment.get('name') == 'energy':
                        if valeur_num == 0:
                            return True
                    elif nutriment.get('name') == 'sugars':
                        if valeur_num < 0 or valeur_num > 100:
                            return True
                except(ValueError, TypeError):
                    return False
            else:
                return False
                
    return False

df_filtre = df[df['nutriments'].apply(lambda x : verif_value_100g(x))]
print(len(df))
print(len(df_filtre))
print(f"Nombre de lignes qui n'ont aucun problème apparent : {len(df) - len(df_filtre)}")

In [ ]:
# Rayons couverts :

# - product_name,

# - code,

# - nutriments,

# - categories_tags

# - brands,

# - images,

# - nutriscore_grade

# - nutriscore_score

# Seuil de complétude : Le produit est conservé SI ET SEULEMENT SI les colonnes code, product_name, et l'énergie dans nutriments sont présentes à 100%"

# TP 3

*voir ~CONTRIBUTING.md*

# TP 4

*voir ~src/load_data.py*

In [ ]:

df_brands = df[['brands']][df['brands'].notna()].drop_duplicates()
df_brands_list = df_brands.values.tolist()
df_brands_list

In [ ]:
food_light_parquet = '../../data/food_light.parquet'
df = pd.read_parquet(food_light_parquet)

# 4. Préparation et insertion pour la table BRANDS
print("Insertion des marques...")
df_brands = df[['brands']][df['brands'].notna()].drop_duplicates()
df_brands_list = df_brands.values.tolist()
df_brands_list

## Peut être lancé indépendamment des autres

In [1]:
import psycopg2
import pandas as pd
import numpy as np
from psycopg2 import extras

def reset_database(cursor):
    """Réinitialise la base de données (Idempotence)"""
    print("Réinitialisation de la base de données...")
    sql_file_path = '../../sql/postgre_creation.sql'
    with open(sql_file_path, 'r', encoding='utf-8') as file :
        sql_script = file.read()
        cursor.execute(sql_script)
    pass

conn = psycopg2.connect(
    dbname="nutriscope", 
    user="postgres", 
    password="postgres", 
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

# 2. Idempotence : on recrée les tables
reset_database(cursor)
conn.commit()

Réinitialisation de la base de données...


In [2]:
# 3. Lecture des données Pandas
print("Chargement du fichier Parquet...")
food_light_parquet = '../../data/food_column_filtered.parquet'
df = pd.read_parquet(food_light_parquet) # 1min48,7s - 1min09 & 51s - 57s - 1:32 MacOS

Chargement du fichier Parquet...


In [3]:
# 4. Préparation et insertion pour la table BRANDS
print("Insertion des marques...")
df_brands = (df['brands'][(df['brands'].notna()) & ( df['brands'] != '')]
             .astype(str)
             .str.capitalize()
             .str.replace("\x00", "", regex=False) # Ici, regex=False fonctionne !
             .drop_duplicates()).replace('Nan', 'Unknown')

print(df_brands)
# - Rédige la requête INSERT appropriée
tuples_brands = [(brand,)for brand in df_brands]
query_brands = "INSERT INTO brands (name) VALUES %s ON CONFLICT (name) DO NOTHING"
extras.execute_values(cursor, query_brands, tuples_brands)
conn.commit() # SELECT COUNT(*) FROM brands : 416_271 / 425_784

Insertion des marques...
0                         Bovetti
1                          Lagg's
13                 Canola harvest
16            Today's temptations
20                       Milkyway
                    ...          
4636422    Wainwright united bees
4636425        Humboldt cider co.
4636451                Tita frita
4636456         Asia home gourmet
4636458        Ocean's favourites
Name: brands, Length: 416271, dtype: str


In [4]:
# Enlever les lignes avec des codes en doublons
df = df.drop_duplicates('code')

In [5]:
df_copy = df.explode('categories_tags').copy()
df_copy = df_copy[df_copy['categories_tags'].str.startswith('en:')].dropna()
print(df_copy['categories_tags'].nunique()) # **:... 105_540 en:... 32_439 / 33_459 (MacOS)

32439


In [6]:
df_copy['categories_tags']

0                 en:breakfasts
0                    en:spreads
0              en:sweet-spreads
0           en:hazelnut-spreads
0          en:chocolate-spreads
                   ...         
4636434          en:Dagligvarer
4636436          en:Dagligvarer
4636437          en:Dagligvarer
4636438          en:Dagligvarer
4636447       en:Pork cracklins
Name: categories_tags, Length: 6284008, dtype: str

In [7]:
# 5. Préparation et insertion pour CATEGORIES
df_categories = df_copy['categories_tags'].unique()

In [8]:
print(df_categories) # 33_459 catégories
print(df_categories[1])
print(df_categories[1][3:]) # [3:] afin de retirer les 3 premières lettres qui sont 'en:' 'fr:' 'de:' ...

<ArrowStringArray>
[                     'en:breakfasts',                         'en:spreads',
                   'en:sweet-spreads',                'en:hazelnut-spreads',
               'en:chocolate-spreads',     'en:cocoa-and-hazelnuts-spreads',
 'en:plant-based-foods-and-beverages',                       'en:beverages',
                   'en:hot-beverages',           'en:plant-based-beverages',
 ...
                     'en:pig-carcass',                       'en:Musztardy',
                          'en:Kapary',                     'en:Pentry Food',
                 'en:Soft Drink Cola',                        'en:Cetrioli',
                   'en:Kwas chlebowy',               'en:Chocolate dotties',
                    'en:whey protein',                  'en:Pork cracklins']
Length: 32439, dtype: str
en:spreads
spreads


In [9]:
# Envoi en DB
print("Insertion des catégories...")
tuples_categories = [(category[3:],)for category in df_categories]
query_categories = "INSERT INTO categories (name) VALUES %s ON CONFLICT (name) DO NOTHING"
extras.execute_values(cursor, query_categories, tuples_categories)
conn.commit() # 0.3s - 33_459 : j'ai constaté qu'il y avait des catégories 'null' 'Undefined'

Insertion des catégories...


### RESET ICI

############################################

In [10]:
df_products = df.copy()

############################################

In [11]:
print("Récupération des marques")
brands_query = 'SELECT * FROM brands'
cursor.execute(brands_query)
brands_tuple = cursor.fetchall()
brands_tuple # (id, name)

Récupération des marques


[(1, 'Bovetti'),
 (2, "Lagg's"),
 (3, 'Canola harvest'),
 (4, "Today's temptations"),
 (5, 'Milkyway'),
 (6, "Mcvitie's"),
 (7, "Sharwood's"),
 (8, 'Allfitnessfactory.de'),
 (9, 'Tetley,  american power products  inc.'),
 (10, 'La fournée campanière'),
 (11, 'Wise woodworks'),
 (12, 'Intermarché'),
 (13, "Goode's bakery & co.  inc."),
 (14, 'Mt. olive'),
 (15, "Walton's flies"),
 (16, 'Eco-dent'),
 (17, 'Lotus brands  inc.'),
 (18, 'Ritter sport,  alfred ritter gmbh & co'),
 (19, 'Ritter sport'),
 (20, 'Ritter sport,  alfred ritter gmbh & co. kg'),
 (21, 'Petpro products  inc.'),
 (22, "Bart & judy's"),
 (23, 'Lactaid'),
 (24, 'Chio'),
 (25, 'Lindt,  polo leathergoods'),
 (26, 'Genius'),
 (27, 'Terres et céréales bio'),
 (28, 'Madelaine chocolate novelties'),
 (29, 'The madelaine chocolate company'),
 (30, 'Madelaine chocolate novelties  inc'),
 (31, 'Innovative candy concepts'),
 (32, 'Ocean mist farms'),
 (33, 'Cean mist farms'),
 (34, 'Ocean mist'),
 (35, 'Better ideas'),
 (36, 'Kee

In [12]:
type(brands_tuple)
brands_pd_df = pd.DataFrame(brands_tuple, columns=['brand_id', 'brand_name'])
brands_pd_df

,brand_id,brand_name
0,1,Bovetti
1,2,Lagg's
2,3,Canola harvest
3,4,Today's temptations
4,5,Milkyway
...,...,...
416265,416267,Wainwright united bees
416266,416268,Humboldt cider co.
416267,416269,Tita frita
416268,416270,Asia home gourmet


In [13]:
print(type(df_products))
print(len(df_products))
print(type(brands_pd_df))
print(len(brands_pd_df))

<class 'pandas.DataFrame'>
4636411
<class 'pandas.DataFrame'>
416270


In [14]:
df_products_brands_merge = df_products.merge(brands_pd_df, how="left", left_on='brands', right_on='brand_name') # left pour récupérer les 4.7M

In [15]:
df_products

,code,brands,product_name,nutriments,categories_tags,nutriscore_score
0,0000101209159,Bovetti,"[{'lang': 'main', 'text': 'Véritable pâte à ta...","[{'name': 'fruits-vegetables-nuts', 'value': N...","[en:breakfasts, en:spreads, en:sweet-spreads, ...",25.0
1,0000105000011,Lagg's,"[{'lang': 'main', 'text': 'Chamomile Herbal Te...",[{'name': 'fruits-vegetables-legumes-estimate-...,[en:null],NaN
2,0000105000042,Lagg's,"[{'lang': 'main', 'text': 'Lagg's, herbal tea,...","[{'name': 'sodium', 'value': 4.0, '100g': 0.00...","[en:plant-based-foods-and-beverages, en:bevera...",NaN
3,0000105000059,Lagg's,"[{'lang': 'main', 'text': 'Linden Flowers Tea'...","[{'name': 'energy-kcal', 'value': 213.32000732...","[en:beverages-and-beverages-preparations, en:p...",NaN
4,0000105000073,Lagg's,"[{'lang': 'main', 'text': 'Herbal Tea, Hibiscu...","[{'name': 'energy', 'value': 267.0, '100g': 11...",None,NaN
...,...,...,...,...,...,...
4636466,5940477600233,NaN,[],None,None,NaN
4636467,8906009532905,NaN,[],None,None,NaN
4636468,2516034005082,NaN,"[{'lang': 'main', 'text': 'unicarm c-ti afumat...",None,None,NaN
4636469,5941896400107,NaN,[],None,[],NaN


In [16]:
df_test = df_products.copy()
df_test.columns

Index(['code', 'brands', 'product_name', 'nutriments', 'categories_tags',
       'nutriscore_score'],
      dtype='str')

In [17]:
df_product_name = df_test[['code','product_name']]
df_product_name.columns

Index(['code', 'product_name'], dtype='str')

In [18]:
print("Test fonction")
# 1. Création d'un DataFrame d'exemple avec votre structure
# df_test = pd.DataFrame({
#     'traductions': [
#         np.array([{'lang': 'main', 'text': 'Chamomile Herbal Tea'}, {'lang': 'en', 'text': 'Chamomile Tea'}], dtype=object),
#         np.array([{'lang': 'fr', 'text': 'Thé à la camomille'}], dtype=object), # Pas de 'main' ici
#         np.array([], dtype=object) # Cas vide pour la sécurité
#     ]
# })

# 2. Fonction de filtrage
def extraire_lang_et_text(tableau):
    # Sécurité si la cellule est vide ou n'est pas un tableau/liste
    if not isinstance(tableau, (list, np.ndarray)) or len(tableau) == 0:
        return pd.Series({'lang': None, 'text': None})
    
    # Étape 1 : Chercher si 'main' existe
    for item in tableau:
        if item.get('lang') == 'fr':
            return pd.Series({'lang': item.get('lang'), 'text': item.get('text')})
        else :
            if item.get('lang') == 'main':
                return pd.Series({'lang': item.get('lang'), 'text': item.get('text')})
    
    # Étape 2 : Si 'main' n'existe pas, récupérer le premier élément présent
    premier_item = tableau[0]
    return pd.Series({'lang': premier_item.get('lang'), 'text': premier_item.get('text')})

# 3. Application de la fonction pour générer deux nouvelles colonnes propres
# df_test[['lang', 'text']] = df_test['traductions'].apply(extraire_lang_et_text)

# print(df_test[['lang', 'text']])

Test fonction


In [19]:
print(df_products.columns)
df_products_name = df_products[['code', 'product_name']].copy()
df_products_name = df_products_name.explode('product_name')

Index(['code', 'brands', 'product_name', 'nutriments', 'categories_tags',
       'nutriscore_score'],
      dtype='str')


In [20]:
df_products_name['lang'] = df_products_name['product_name'].str.get('lang')
df_products_name['name'] = df_products_name['product_name'].str.get('text')